# LCEL 체인에 대화 기록 추가하기

> 업데이트 기준: 2026-09 · LangChain 1.x / LangGraph 1.x

`RunnableWithMessageHistory`도 현재 버전에서는 deprecated 경고를 냅니다. 새 코드에서는
LCEL runnable을 LangGraph 노드로 사용하고, 그래프에 checkpointer를 연결하는 방식이
권장됩니다.

이 방식은 LCEL의 `prompt | model` 조합을 그대로 유지하면서도 대화 상태, 스레드 격리,
중단·재개와 영속 저장을 LangGraph의 한 가지 메커니즘으로 통일합니다.


In [1]:
# 필요한 경우 아래 줄의 주석을 해제하고 한 번만 실행하세요.
# %pip install -qU "langchain>=1.0" "langchain-openai>=1.0" "langgraph>=1.0" python-dotenv


In [2]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# 다른 공급자를 쓸 때는 예: anthropic:claude-... 처럼 지정할 수 있습니다.
MODEL_ID = os.getenv("CHAT_MODEL", "openai:gpt-5.6-luna")
model = init_chat_model(MODEL_ID)


## 메시지 목록을 입력받는 LCEL 체인

현재 사용자 메시지를 별도 문자열로 분리하지 않고, 그래프 상태의 전체 `messages`를
`MessagesPlaceholder`에 전달합니다.


In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절하고 간결한 챗봇입니다."),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

lcel_chain = prompt | model


## LCEL runnable을 LangGraph 노드로 감싸기

노드는 LCEL 체인의 새 AI 메시지만 반환합니다. `MessagesState`의 reducer가 이를 기존
메시지 뒤에 추가하고, checkpointer가 `thread_id`별 상태를 저장합니다.


In [4]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph


def call_lcel(state: MessagesState):
    response = lcel_chain.invoke({"messages": state["messages"]})
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("lcel", call_lcel)
builder.add_edge(START, "lcel")
builder.add_edge("lcel", END)

chat = builder.compile(checkpointer=InMemorySaver())


In [5]:
config = {"configurable": {"thread_id": "teddy-session"}}

result = chat.invoke(
    {"messages": [{"role": "user", "content": "만나서 반갑습니다. 제 이름은 테디입니다."}]},
    config=config,
)
print(result["messages"][-1].content)


만나서 반가워요, 테디님! 무엇을 도와드릴까요?


In [6]:
result = chat.invoke(
    {"messages": [{"role": "user", "content": "제 이름이 무엇이었는지 기억하세요?"}]},
    config=config,
)
print(result["messages"][-1].content)


네, 기억해요. 이름은 **테디**입니다!


checkpointer가 사용자 입력과 모델 응답을 모두 기록했는지 확인합니다.


In [7]:
for message in chat.get_state(config).values["messages"]:
    print(f"[{message.type}] {message.content}")


[human] 만나서 반갑습니다. 제 이름은 테디입니다.
[ai] 만나서 반가워요, 테디님! 무엇을 도와드릴까요?
[human] 제 이름이 무엇이었는지 기억하세요?
[ai] 네, 기억해요. 이름은 **테디**입니다!


다른 `thread_id`는 새로운 기록을 사용합니다.


In [8]:
other_config = {"configurable": {"thread_id": "new-session"}}
result = chat.invoke(
    {"messages": [{"role": "user", "content": "제 이름이 무엇이었나요?"}]},
    config=other_config,
)
print(result["messages"][-1].content)


아직 이름을 알려주시지 않았어요.
